# PART 2 — Reading & Writing Data 
### PySpark on Databricks — Complete Learning Series

**How to use this notebook:**
1. `Workspace -> Import -> File` → upload this `.py` file (auto-detected as a notebook).
2. Attach to any cluster (works on **Databricks Community Edition** too — call-outs below mark anything that needs a paid workspace).
3. Run top-to-bottom with `Shift+Enter`. This notebook is **self-contained** — it generates its own
   sample CSV/JSON/Parquet/ORC/Text files first, so you don't need to upload anything.
4. Practice Questions + Solutions at the end.

**Topics covered in Part 2:**
10. Reading CSV, JSON, Parquet, ORC, Text files
11. Read options (header, inferSchema, delimiter, multiline) + modes (PERMISSIVE/DROPMALFORMED/FAILFAST) + error handling
12. Writing files (CSV, JSON, Parquet) with modes: overwrite, append, ignore, error
13. Partitioned read/write (`partitionBy`)
14. Reading from JDBC (SQL databases)
15. Reading from Delta Lake tables
16. DBFS paths, mounting cloud storage (ADLS/S3), Unity Catalog volumes
17. 🎯 Practice Questions with Solutions

> ⚠️ **Databricks Community Edition (CE) note (read this first):**
> CE gives you a free single-node cluster with DBFS storage, and **Delta Lake works fully**.
> However CE does **NOT** support: **Unity Catalog**, mounting your own cloud storage account
> (ADLS/S3) since you have no cloud credentials tied to it, and some Jobs/cluster-scaling features.
> Everywhere this matters, you'll see a boxed **"CE ALTERNATIVE"** note telling you exactly what
> to use instead so every example in this notebook still runs on CE.

## 0. Setup — Generate Sample Files to Practice With
We create one Employee dataset and write it out in every format, so every "read" example
below has a real file to read back. This makes the notebook 100% runnable anywhere.

DButils

In [0]:
# 2) Create a folder in DBFS (like "mkdir") - useful to organize your own practice files
dbutils.fs.mkdirs("/Volumes/ml/ai_schema/somedatavolume/new")      # creates a folder (no error if it already exists)
display(dbutils.fs.ls("/Volumes/ml/ai_schema/somedatavolume/"))          # confirm the folder was created

In [0]:
# 3) Widgets: create interactive input parameters at the top of your notebook (super useful for reusable notebooks / jobs)
dbutils.widgets.text("dept_filter", "IT")              # creates a text-box widget named "dept_filter" with default value "IT"
selected_dept = dbutils.widgets.get("dept_filter")      # read the current value typed by the user in the widget box
print("You selected department:", selected_dept)
# dbutils.widgets.remove("dept_filter")                 # uncomment to remove the widget when done

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Base folder for all Part 2 practice files (works on CE and full Databricks alike - plain DBFS path)
base_path = "/Volumes/pysparkcatalog/pyspark_schema/files/pyspark_part2_sample_files/write/"
dbutils.fs.mkdirs(base_path)          # create the folder (no error if it already exists)

# Our sample Employee data (reused throughout Part 2)
employee_data = [
    (1, "Amit",  "IT",     55000, 28, "Pune"),
    (2, "Sneha", "HR",     48000, 32, "Mumbai"),
    (3, "Raj",   "Sales",  62000, 25, "Delhi"),
    (4, "Priya", "IT",     71000, 30, "Pune"),
    (5, "Karan", "Sales",  39000, 27, "Delhi"),
    (6, "Neha",  "HR",     54000, 29, "Mumbai"),
    (7, "Vikas", "IT",     67000, 35, "Bangalore"),
    (8, "Meena", "Sales",  45000, 24, "Delhi"),
]
employee_columns = ["id", "name", "department", "salary", "age", "city"]
df_source = spark.createDataFrame(employee_data, employee_columns)
df_source.show()

# Write this ONE dataframe out to CSV, JSON, Parquet, ORC, and plain Text so we have files to read in Section 1.
# df_source.write.mode("overwrite").option("header", True).csv(f"{base_path}/employees_csv")
# df_source.write.mode("overwrite").json(f"{base_path}/employees_json")
# df_source.write.mode("overwrite").parquet(f"{base_path}/employees_parquet")
# df_source.write.mode("overwrite").orc(f"{base_path}/employees_orc")

df_source.selectExpr("concat_ws(',', id, name, department, salary, age, city) as line") \
         .write.mode("overwrite").text(f"{base_path}/employees_text")   # text() needs a SINGLE string column

print("Sample files created under:", base_path)
display(dbutils.fs.ls(base_path))

ORC format:ORC (which stands for Optimized Row Columnar) is a free, open-source, column-oriented data storage format.

id column
---------
1
2
3

name column
-----------
Amit
Neha
Rahul

salary column
-------------
50000
54000
70000

## 10. Reading CSV, JSON, Parquet, ORC, Text Files

```
  File Format Cheat Sheet
  ┌───────────┬────────────────────────────┬──────────────────────────────────┐
  │  Format   │  Stores schema internally?  │  Best use case                    │
  ├───────────┼────────────────────────────┼──────────────────────────────────┤
  │  CSV      │  No (plain text, row/col)   │  Excel exports, simple data dumps │
  │  JSON     │  No (self-describing keys)  │  Semi-structured / nested data     │
  │  Parquet  │  YES (columnar + schema)    │  Big data analytics (FAST, default)│
  │  ORC      │  YES (columnar + schema)    │  Hive-heavy environments            │
  │  Text     │  No (one column per line)   │  Logs, unstructured raw text        │
  │  Delta    │  YES (+ transaction log)    │  Production lakehouse tables (Part 15)│
  └───────────┴────────────────────────────┴──────────────────────────────────┘
```
**Real-life analogy:** CSV/Text are like a plain hand-written list — readable but Spark must
guess the structure. Parquet/ORC are like a pre-labeled, pre-organized filing cabinet —
Spark can jump straight to the columns/rows it needs without reading everything (this is
called **columnar storage** + **predicate pushdown**), making them dramatically faster.

In [0]:
base_path = "/Volumes/pysparkcatalog/pyspark_schema/files/pyspark_part2_sample_files"

In [0]:
# ---- 10.1 Reading CSV ----
df_csv = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/employees.csv")
df_csv.show()
df_csv.printSchema()
#inferSchema tells Spark to automatically detect the data type of each column instead of reading everything as a string.

In [0]:
# ---- 10.2 Reading JSON ----
df_json = spark.read.json(f"{base_path}/employees.json")   # JSON is self-describing -> schema inferred automatically, no options needed
df_json.show()
df_json.printSchema()

In [0]:
# ---- 10.3 Reading Parquet (schema is stored INSIDE the file - no inferSchema/header options needed at all) ----
df_parquet = spark.read.parquet(f"{base_path}/employees.parquet")
df_parquet.show()
df_parquet.printSchema()          # notice: correct types (int, double) came back automatically - Parquet remembers them!

In [0]:
# ---- 10.4 Reading ORC ----
df_orc = spark.read.orc(f"{base_path}/employees_sample.orc")
df_orc.show()

In [0]:
# ---- 10.5 Reading plain Text (every line becomes ONE row in a single column called "value") ----
df_text = spark.read.text(f"{base_path}/employees.txt")
df_text.show(truncate=False)
df_text.printSchema()             # always just one column: "value" (StringType)

# You'd typically split this manually afterward:
from pyspark.sql.functions import split, col
df_text_split = df_text.withColumn("parts", split(col("value"), ","))
df_text_split.select(
    col("parts")[0].alias("id"),
    col("parts")[1].alias("name"),
    col("parts")[2].alias("department")
).show()

In [0]:
# ---- 10.6 Generic reader syntax (format() + load()) — works for ANY format, useful when the format is a variable ----
df_generic = spark.read.format("csv").option("header", True).option("inferSchema", True).load(f"{base_path}/employees_csv")
df_generic.show(3)
# This is EXACTLY equivalent to spark.read.csv(...) - "format().load()" is just the more general/explicit syntax,
# and it's the ONLY way to read some formats like "delta", "jdbc", "avro".

## 11. Read Options + Modes (PERMISSIVE / DROPMALFORMED / FAILFAST) + Error Handling

### Common read options
| Option | Purpose | Example |
|---|---|---|
| `header` | First row is column names? (CSV) | `.option("header", True)` |
| `inferSchema` | Auto-detect column data types (CSV) — slower on big files, scans data twice | `.option("inferSchema", True)` |
| `delimiter` / `sep` | Character separating columns (CSV) | `.option("delimiter", ";")` |
| `multiline` | Allow a single JSON/CSV record to span multiple lines | `.option("multiline", True)` |
| `quote` | Character used to wrap values containing the delimiter | `.option("quote", "\"")` |
| `escape` | Escape character inside quoted values | `.option("escape", "\\")` |
| `dateFormat` | Pattern to parse date strings | `.option("dateFormat", "yyyy-MM-dd")` |
| `nullValue` | String that should be treated as NULL | `.option("nullValue", "NA")` |
| `mode` | How to handle BAD/malformed records (see below) | `.option("mode", "PERMISSIVE")` |
| `columnNameOfCorruptRecord` | Column to store the raw bad record when mode=PERMISSIVE | `.option("columnNameOfCorruptRecord","_corrupt")` |

### The 3 parsing modes (CSV & JSON)
```
  PERMISSIVE (DEFAULT)
    - Keeps ALL rows, sets fields it couldn't parse to NULL,
      and (optionally) stores the raw bad line in a "_corrupt_record" column.
    - Best for: exploring messy data without losing any rows.

  DROPMALFORMED
    - SILENTLY DROPS any row that doesn't match the schema. No error, no trace.
    - Best for: quick analysis where a few bad rows genuinely don't matter.
    - ⚠ Risk: you can silently lose data without realizing it - use with caution.

  FAILFAST
    - THROWS AN EXCEPTION immediately the moment it hits ANY malformed record.
    - Best for: production pipelines where bad data must STOP the job (data quality gate).
```
**Real-life analogy:** Think of a security guard checking IDs at a club entrance.
- PERMISSIVE = lets everyone in, but flags the fake IDs on a clipboard.
- DROPMALFORMED = quietly turns away anyone with a bad ID, no questions asked.
- FAILFAST = shuts down the entire club the second ONE fake ID is spotted.

In [0]:
# ---- Create a MESSY csv file on purpose (with a bad row) so we can practice error handling ----

base_path = "/Volumes/pysparkcatalog/pyspark_schema/files/pyspark_part2_sample_files"

messy_csv_content = """id,name,department,salary
1,Amit,IT,55000
2,Sneha,HR,48000
BAD_ROW_HERE_ONLY_TWO_FIELDS,IT,67.005
4,Priya,IT,71000"""

dbutils.fs.put(f"{base_path}/messy_data2.csv", messy_csv_content, overwrite=True)   # write raw text directly to DBFS

In [0]:
base_path = "/Volumes/pysparkcatalog/pyspark_schema/files/pyspark_part2_sample_files/messy_data2.csv"
df=spark.read.option("header", True).csv(base_path)
df.display()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
#---- 11.1 PERMISSIVE mode (default): bad row survives as NULLs + optionally captured ----

messy_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True),   # extra column to CATCH the raw bad line
])

df_permissive = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")   # tell Spark WHERE to store the raw bad line
    .schema(messy_schema)                                       # NOTE: columnNameOfCorruptRecord requires an explicit schema!
    .csv(f"{base_path}")
)
df_permissive.show(truncate=False)   # the bad row shows as NULL fields + its raw text in "_corrupt_record"

In [0]:
# ---- 11.2 DROPMALFORMED mode: bad row silently disappears ----
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
df_dropmalformed = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "DROPMALFORMED")
    .csv(f"{base_path}")
)
df_dropmalformed = df_dropmalformed.withColumn(
    "id", col("id").cast("int")
)
df_dropmalformed.show()   # only 3 GOOD rows remain - the bad row is gone with no warning printed

In [0]:
# ---- 11.3 FAILFAST mode: throws an exception -> wrap in try/except for proper error handling ----
try:
    df_failfast = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("mode", "FAILFAST")
        .csv(f"{base_path}")
    )
    df_failfast.show()          # this line will NEVER be reached if a bad row exists
except Exception as e:
    print("❌ FAILFAST caught a bad record! Stopping pipeline as expected.")
    print("Error detail (truncated):", str(e)[:300])   # print first 300 chars of the error message

### 11.4 More error-handling patterns for production pipelines

In [0]:
# Pattern A: Defensive read — wrap ANY read in try/except so one bad file doesn't crash your whole job
def safe_read_csv(path):
    """Reads a CSV safely; returns None (and logs the error) instead of crashing the whole notebook/job."""
    try:
        return spark.read.option("header", True).option("inferSchema", True).csv(path)
    except Exception as err:
        print(f"⚠️ Failed to read {path}: {err}")
        return None

result_df = safe_read_csv(f"{base_path}")
if result_df is not None:
    result_df.show(3)

In [0]:
# Pattern B: Isolate & inspect bad records using PERMISSIVE mode + _corrupt_record, then quarantine them
good_records = df_permissive.filter(col("_corrupt_record").isNull()).drop("_corrupt_record")
bad_records  = df_permissive.filter(col("_corrupt_record").isNotNull())

print("✅ Good records:")
good_records.show()
print("❌ Bad/quarantined records (send these to a review folder or alert a data-quality dashboard):")
bad_records.show(truncate=False)

In [0]:
# ---- Multiline JSON example (a JSON record that spans several lines - common in exported/pretty-printed JSON) ----
multiline_json = """[
  {
    "id": 1,
    "name": "Amit",
    "department": "IT"
  },
  {
    "id": 2,
    "name": "Sneha",
    "department": "HR"
  }
]"""
dbutils.fs.put(f"{base_path}/multiline.json", multiline_json, overwrite=True)

# WITHOUT multiline=True, Spark expects ONE JSON object PER LINE and this file would fail/return garbage.
df_multiline = spark.read.option("multiline", True).json(f"{base_path}/multiline.json")
df_multiline.show()

In [0]:
# ---- Custom delimiter example (e.g. pipe-separated values) ----
base_path = "/Volumes/pysparkcatalog/pyspark_schema/files/pyspark_part2_sample_files/"
pipe_data = "id|name|department\n1|Amit|IT\n2|Sneha|HR"
dbutils.fs.put(f"{base_path}/pipe_data.csv", pipe_data, overwrite=True)

df_pipe = spark.read.option("header", True).option("delimiter", "|").option("inferSchema", True).csv(f"{base_path}/pipe_data.csv")
df_pipe.show()

## 12. Writing Files (CSV, JSON, Parquet) with Modes: overwrite, append, ignore, error

| Mode | Behavior | Real-life analogy |
|---|---|---|
| `overwrite` | DELETES existing data at the path and writes fresh | Erasing a whiteboard before writing new notes |
| `append` | ADDS new data to whatever already exists at the path | Adding a new page to an existing notebook |
| `ignore` | If data ALREADY exists at the path, do NOTHING (no error, no write) | "Don't overwrite my file if it's already there" |
| `error` / `errorifexists` (DEFAULT) | Throws an EXCEPTION if the path already has data | A safety lock preventing accidental overwrites |

In [0]:
# ============================================================
# WRITE MODE: error / errorifexists
# ============================================================

# 1. Create sample DataFrame
data = [
    (1, "Amit", "IT", 55000),
    (2, "Sneha", "HR", 48000),
    (3, "Rahul", "Finance", 62000),
    (4, "Priya", "IT", 71000)
]

columns = ["id", "name", "department", "salary"]

df_source = spark.createDataFrame(data, columns)

print("Source Data:")
df_source.show()


# 2. Define paths
base_path = "/Volumes/pysparkcatalog/pyspark_schema/files/pyspark_part2_sample_files"
output_path = f"{base_path}/write_demo"


# 3. Optional cleanup - makes the cell runnable again from scratch
# dbutils.fs.rm(f"{output_path}_v1", True)


# 4. FIRST WRITE
# "error" is also Spark's default write mode.
# It works because the path does not exist yet.

df_source.write \
    .mode("error") \
    .option("header", True) \
    .csv(f"{output_path}_v1")

print("✅ First write successful")


# 5. SECOND WRITE to the SAME path
# This fails because the output path already exists.

try:
    df_source.write \
        .mode("error") \
        .option("header", True) \
        .csv(f"{output_path}_v1")

except Exception as e:
    print("❌ 'error' mode blocked the second write as expected")
    print("Error:", str(e)[:500])


# 6. Read the successfully written CSV files
df_result = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{output_path}_v1")

print("Data already present in output path:")
df_result.show()

In [0]:
# ---- ignore ----
df_source.write.mode("ignore").csv(f"{output_path}_v1")     # path already exists from above -> WRITE IS SILENTLY SKIPPED
print("Row count still same as before (write was skipped):", spark.read.csv(f"{output_path}_v1").count())

In [0]:
# ---- overwrite ----
df_half = df_source.filter(col("department") == "IT")        # only 2 rows
df_half.write.mode("overwrite").option("header", True).csv(f"{output_path}_v1")
print("Row count after OVERWRITE with only IT dept:", spark.read.option("header", True).csv(f"{output_path}_v1").count())
df_read=spark.read.option("header", True).csv(f"{output_path}_v1")
df_read.show()

In [0]:
# ---- append ----
df_more = df_source.filter(col("department") == "HR")        # 3 more rows
df_more.write.mode("append").option("header", True).csv(f"{output_path}_v1")
print("Row count after APPEND (IT rows + HR rows):", spark.read.option("header", True).csv(f"{output_path}_v1").count())
df_read=spark.read.option("header", True).csv(f"{output_path}_v1")
df_read.show()

In [0]:
# ---- Writing JSON and Parquet with modes works identically ----
df_source.write.mode("overwrite").json(f"{output_path}_json")
df_source.write.mode("overwrite").parquet(f"{output_path}_parquet")

# Compression option (Parquet defaults to snappy already, but you can control it)
df_source.write.mode("overwrite").option("compression", "gzip").parquet(f"{output_path}_parquet_gzip")
print("Files written with different compression codecs - check file sizes:")
display(dbutils.fs.ls(f"{output_path}_parquet"))
display(dbutils.fs.ls(f"{output_path}_parquet_gzip"))

In [0]:
# ---- coalesce(1) / repartition(1) before writing -> controls HOW MANY output files are created ----
# By default Spark writes ONE FILE PER PARTITION (could be dozens of small files!). For a small demo dataset,
# it's common to force a SINGLE output file so it's easy to share/inspect.
df_source.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{output_path}_single_file")
display(dbutils.fs.ls(f"{output_path}_single_file"))   # notice: one part-0000....csv file (plus a _SUCCESS marker file)

## 13. Partitioned Read/Write (`partitionBy`)

**What is partitioning on write?** Instead of one giant file, Spark creates a **folder per
unique value** of the column(s) you choose. This lets future reads SKIP entire folders that
don't match a filter (called **partition pruning**) — massively speeding up queries on huge datasets.

**Real-life analogy:** Instead of keeping all invoices in one giant pile, you file them into
labeled drawers by "Year" and "Month". Looking for March 2024 invoices? Open ONE drawer,
ignore the rest — instead of flipping through the entire pile.

```
  employees_partitioned/
      ├── department=IT/
      │      └── part-0000.parquet
      ├── department=HR/
      │      └── part-0000.parquet
      └── department=Sales/
             └── part-0000.parquet
```

In [0]:
# ---- Writing partitioned by ONE column ----
partitioned_path = f"{base_path}/employees_partitioned"
df_source.write.mode("overwrite").partitionBy("department").parquet(partitioned_path)

display(dbutils.fs.ls(partitioned_path))   # notice folders: department=IT, department=HR, department=Sales

In [0]:
# ---- Writing partitioned by MULTIPLE columns (creates nested folders) ----
partitioned_path_multi = f"{base_path}/employees_partitioned_multi"
df_source.write.mode("overwrite").partitionBy("department", "city").parquet(partitioned_path_multi)
display(dbutils.fs.ls(partitioned_path_multi))                       # top level: department=IT, department=HR...
display(dbutils.fs.ls(f"{partitioned_path_multi}/department=IT"))     # inside IT: city=Pune, city=Bangalore...

In [0]:
# ---- Reading a partitioned dataset back ----
df_read_partitioned = spark.read.parquet(partitioned_path)
df_read_partitioned.printSchema()      # "department" comes back as a normal column even though it was a FOLDER name!
df_read_partitioned.show()

# ---- Partition pruning in action: filtering on the partition column is SUPER fast (skips other folders entirely) ----
df_it_only = spark.read.parquet(partitioned_path).filter(col("department") == "IT")
df_it_only.explain()      # look for "PartitionFilters" in the plan -> proof Spark skipped HR/Sales folders on disk
df_it_only.show()

## 14. Reading from JDBC (SQL Databases)

JDBC lets Spark connect DIRECTLY to relational databases (MySQL, PostgreSQL, SQL Server,
Oracle, etc.) and read/write tables as DataFrames.

**Requirements:**
1. The database's **JDBC driver JAR** must be installed on the cluster (`Compute -> Libraries -> Install New -> Maven`, search e.g. `mysql:mysql-connector-java`).
2. Network connectivity from the cluster to the database (firewall/VPC rules).
3. A valid connection URL, username, and password (use **Databricks Secrets**, never hardcode passwords!).

> ⚠️ **Community Edition note:** CE clusters CAN install JDBC driver JARs and CAN reach public
> internet databases, so JDBC reads DO work on CE — **as long as your database allows inbound
> connections from Databricks' cloud IP range** (most personal/local databases behind a home
> router will NOT be reachable). For practicing without a real database, use a small public
> demo DB, or simply skip to the code pattern below and swap in your own working credentials later.

In [0]:
# ---- Standard JDBC read pattern (syntax reference - replace placeholders with your real DB details) ----

jdbc_url = "jdbc:mysql://<host>:<port>/<database>"     # e.g. jdbc:mysql://mydb.company.com:3306/sales_db

connection_properties = {
    "user": dbutils.secrets.get(scope="my-scope", key="db-username"),   # NEVER hardcode credentials - use Secrets!
    "password": dbutils.secrets.get(scope="my-scope", key="db-password"),
    "driver": "com.mysql.cj.jdbc.Driver"                                  # driver class name (varies per DB vendor)
}

# Reading an ENTIRE table:
# df_jdbc = spark.read.jdbc(url=jdbc_url, table="employees", properties=connection_properties)

# Reading with a CUSTOM SQL QUERY instead of a full table (push filtering down to the DB - very efficient):
# query = "(SELECT id, name, salary FROM employees WHERE department = 'IT') AS subquery"
# df_jdbc_query = spark.read.jdbc(url=jdbc_url, table=query, properties=connection_properties)

# Alternative syntax using format()/option() (equivalent to the above, just more explicit):
# df_jdbc2 = (
#     spark.read.format("jdbc")
#     .option("url", jdbc_url)
#     .option("dbtable", "employees")
#     .option("user", "myuser")
#     .option("password", "mypassword")
#     .option("driver", "com.mysql.cj.jdbc.Driver")
#     .option("numPartitions", 4)          # read in PARALLEL using 4 tasks (requires partitionColumn/lowerBound/upperBound too)
#     .option("partitionColumn", "id")
#     .option("lowerBound", 1)
#     .option("upperBound", 10000)
#     .load()
# )

print("ℹ️ JDBC code shown above is reference syntax - connect it to your own database to run it.")
print("Parallel JDBC reads (numPartitions/partitionColumn/lowerBound/upperBound) avoid pulling millions of rows through a SINGLE connection.")

### Writing back to a database via JDBC (reference syntax)

In [0]:
# df_source.write.jdbc(
#     url=jdbc_url,
#     table="employees_output",
#     mode="overwrite",                 # same modes as file writes: overwrite / append / ignore / error
#     properties=connection_properties
# )
print("ℹ️ Writing DataFrames to a SQL table uses the exact same .write.mode(...) pattern you already learned in Section 12.")

## 15. Reading from Delta Lake Tables

Delta Lake is an open-source storage layer that adds **ACID transactions, time travel, schema
enforcement, and upserts (MERGE)** on top of Parquet files. It's the DEFAULT table format in
Databricks. (Full deep-dive is in Part 9 — this section covers just reading/writing basics.)

> ✅ **Community Edition note:** Delta Lake is **fully supported on CE** — no limitations here at all!

In [0]:
# ---- Writing a Delta table (path-based) ----
delta_path = f"{base_path}/employees_delta"
df_source.write.format("delta").mode("overwrite").save(delta_path)

# ---- Reading it back ----
df_delta = spark.read.format("delta").load(delta_path)
df_delta.show()

In [0]:
# ---- Registering as a proper managed/external TABLE so you can query it with plain SQL ----
spark.sql(f"CREATE TABLE IF NOT EXISTS employees_delta_tbl USING DELTA LOCATION '{delta_path}'")

# Now query using SQL directly:
spark.sql("SELECT department, COUNT(*) as emp_count, AVG(salary) as avg_salary FROM employees_delta_tbl GROUP BY department").show()

# Or query using the DataFrame API on the TABLE (instead of a path):
spark.table("employees_delta_tbl").show()

In [0]:
# ---- Quick preview of what makes Delta special: DESCRIBE HISTORY (full time-travel is covered in Part 9) ----
spark.sql("DESCRIBE HISTORY employees_delta_tbl").select("version", "timestamp", "operation").show(truncate=False)

## 16. DBFS Paths, Mounting Cloud Storage (ADLS/S3), Unity Catalog Volumes

### DBFS path types you'll see:
| Path style | Meaning |
|---|---|
| `/tmp/...` or `/FileStore/...` | A path INSIDE DBFS (Databricks' built-in distributed file system) |
| `dbfs:/tmp/...` | Same as above, just with the explicit `dbfs:` scheme prefix |
| `/mnt/my-mount/...` | A MOUNT POINT — a shortcut/symlink DBFS keeps to an external cloud storage container |
| `abfss://container@account.dfs.core.windows.net/...` | Direct ADLS Gen2 path (no mount needed, used with Unity Catalog credentials) |
| `/Volumes/catalog/schema/volume_name/...` | A **Unity Catalog Volume** path — the modern replacement for mounts |

### Mounting external cloud storage (legacy pattern, still common in older workspaces)
```python
dbutils.fs.mount(
    source = "abfss://mycontainer@mystorageaccount.dfs.core.windows.net/",
    mount_point = "/mnt/my-adls-mount",
    extra_configs = {
        "fs.azure.account.auth.type": "OAuth",
        "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        "fs.azure.account.oauth2.client.id": dbutils.secrets.get("scope", "client-id"),
        "fs.azure.account.oauth2.client.secret": dbutils.secrets.get("scope", "client-secret"),
        "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/<tenant-id>/oauth2/token"
    }
)
# After mounting, read/write it exactly like any DBFS path:
# spark.read.csv("/mnt/my-adls-mount/some_file.csv")
```

### Unity Catalog Volumes (the modern, governed replacement for mounts)
```python
# Once a Volume is created by an admin:
# spark.read.csv("/Volumes/main_catalog/sales_schema/raw_files/employees.csv")
```
Volumes give fine-grained permissions (who can read/write which folder) governed centrally,
instead of every user needing cloud storage keys.

> ⚠️ **Community Edition note — READ CAREFULLY:**
> - `dbutils.fs.mount(...)` requires a real cloud storage account (Azure/AWS) + credentials — **CE typically has no cloud account attached**, so mounting your own storage generally isn't possible on CE.
> - **Unity Catalog is NOT available on Community Edition at all** — there is no `/Volumes/...` path and no `catalog.schema.table` 3-level namespace; CE only has the legacy single-level `hive_metastore` with `database.table`.
>
> **✅ CE ALTERNATIVES that work everywhere:**
> 1. Use plain **DBFS paths** (`/tmp/...`, `/FileStore/...`) exactly like every example in this notebook does — no mounting needed for practice/learning.
> 2. To bring your OWN files into CE: use the notebook UI **`File -> Upload Data`**, or drag-and-drop into `/FileStore/tables/` via the **Data** tab in the sidebar.
> 3. To pull public sample datasets: Databricks pre-loads many at **`/databricks-datasets/`** (available on CE too) — try `dbutils.fs.ls("/databricks-datasets")`.
> 4. To fetch a file from the internet on CE, use a shell cell: `%sh wget -O /tmp/file.csv <url>` then move it into DBFS with `dbutils.fs.cp("file:/tmp/file.csv", "dbfs:/tmp/file.csv")`.

In [0]:
# ---- Practical DBFS path operations you'll use constantly ----
display(dbutils.fs.ls(base_path))              # list files/folders
dbutils.fs.cp(f"{base_path}/employees_csv", f"{base_path}/employees_csv_backup", recurse=True)  # copy a folder
print("Does backup exist?", any(f.name.startswith("employees_csv_backup") for f in dbutils.fs.ls(base_path)))

# dbutils.fs.rm(f"{base_path}/employees_csv_backup", recurse=True)   # uncomment to delete the backup folder + all contents
# dbutils.fs.head(f"{base_path}/employees_text/part-00000")          # peek at the first bytes of a raw file (uncomment to try)

# 🎯 PRACTICE QUESTIONS — Part 2
Try each question yourself first, then check the solution cell below it.
All questions reuse `df_source` (the Employee dataset) and the `base_path` folder from Section 0.

| Q# | Question | Notes / Expected Result |
|---|---|---|
| 1 | Write `df_source` as JSON to `{base_path}/practice/q1_json`, then read it back and show it | JSON round-trip |
| 2 | Read `messy_data.csv` using `DROPMALFORMED` mode and count how many rows survived | Should be 3 rows |
| 3 | Read `messy_data.csv` using `PERMISSIVE` mode with a `_corrupt_record` column, then show ONLY the corrupt row | 1 row shown |
| 4 | Write `df_source` to `{base_path}/practice/q4_append` twice using `append` mode, then confirm row count doubled | 16 rows |
| 5 | Write `df_source` partitioned by `city` to `{base_path}/practice/q5_partitioned` | Check folder names = city=Pune, city=Mumbai, etc. |
| 6 | Read back the Q5 output and filter for `city = 'Pune'` only, then call `.explain()` to confirm partition pruning | Look for PartitionFilters in the plan |
| 7 | Read the pipe-delimited file `{base_path}/pipe_data.csv` (delimiter `\|`) into a DataFrame | 2 rows, 3 columns |
| 8 | Write `df_source` as a SINGLE output CSV file (use `coalesce`) to `{base_path}/practice/q8_singlefile` | Only 1 part file + _SUCCESS |
| 9 | Write `df_source` as a Delta table at `{base_path}/practice/q9_delta`, then read it back and filter `salary > 50000` | 5 rows |
| 10 | Try writing to `{base_path}/practice/q4_append` using mode `error` and catch the exception gracefully | Should print a caught-error message, not crash |

---
## ✅ SOLUTIONS

In [0]:
# Q1: JSON round-trip
df_source.write.mode("overwrite").json(f"{base_path}/practice/q1_json")
spark.read.json(f"{base_path}/practice/q1_json").show()

In [0]:
# Q2: DROPMALFORMED row count
q2_df = spark.read.option("header", True).option("inferSchema", True).option("mode", "DROPMALFORMED").csv(f"{base_path}/messy_data.csv")
print("Surviving row count:", q2_df.count())
q2_df.show()

In [0]:
# Q3: PERMISSIVE mode - show only the corrupt row
q3_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True),
])
q3_df = (
    spark.read.option("header", True).option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(q3_schema)
    .csv(f"{base_path}/messy_data.csv")
)
q3_df.filter(col("_corrupt_record").isNotNull()).show(truncate=False)

In [0]:
# Q4: Append twice, confirm doubled row count
q4_path = f"{base_path}/practice/q4_append"
df_source.write.mode("overwrite").option("header", True).csv(q4_path)   # first write (8 rows)
df_source.write.mode("append").option("header", True).csv(q4_path)      # second write appends 8 more rows
print("Total rows after 2 writes:", spark.read.option("header", True).csv(q4_path).count())

In [0]:
# Q5: Partition by city
q5_path = f"{base_path}/practice/q5_partitioned"
df_source.write.mode("overwrite").partitionBy("city").parquet(q5_path)
display(dbutils.fs.ls(q5_path))

In [0]:
# Q6: Filter partitioned data + explain() to confirm pruning
q6_df = spark.read.parquet(q5_path).filter(col("city") == "Pune")
q6_df.explain()          # look for "PartitionFilters: [isnotnull(city#.. ), (city#.. = Pune)]" in the output
q6_df.show()

In [0]:
# Q7: Read pipe-delimited file
spark.read.option("header", True).option("delimiter", "|").option("inferSchema", True).csv(f"{base_path}/pipe_data.csv").show()

In [0]:
# Q8: Single output file using coalesce
q8_path = f"{base_path}/practice/q8_singlefile"
df_source.coalesce(1).write.mode("overwrite").option("header", True).csv(q8_path)
display(dbutils.fs.ls(q8_path))    # exactly ONE part-*.csv file + a _SUCCESS marker

In [0]:
# Q9: Delta table write + read + filter
q9_path = f"{base_path}/practice/q9_delta"
df_source.write.format("delta").mode("overwrite").save(q9_path)
spark.read.format("delta").load(q9_path).filter(col("salary") > 50000).show()

In [0]:
# Q10: Catch an 'error' mode exception gracefully
try:
    df_source.write.mode("error").csv(q4_path)     # q4_append path already has data from Q4
except Exception as e:
    print("✅ Caught expected exception - path already exists. Error (truncated):", str(e)[:150])

## ✅ Part 2 Summary — What You Learned
- Reading **CSV, JSON, Parquet, ORC, Text** — and WHY Parquet/ORC (columnar + embedded schema) beat CSV/JSON for big data
- Key **read options**: header, inferSchema, delimiter, multiline, quote/escape, nullValue, dateFormat
- The 3 **parsing modes** — PERMISSIVE (keep + flag), DROPMALFORMED (silently drop), FAILFAST (crash immediately) — and how to pick the right one
- **Error-handling patterns**: try/except wrappers, `_corrupt_record` quarantine pattern, defensive read functions
- **Write modes**: overwrite, append, ignore, error/errorifexists — with live before/after row-count proof
- **partitionBy()** for folder-based partitioning + **partition pruning** proof via `.explain()`
- **JDBC** reference syntax for connecting to relational databases (with Secrets-based credentials, parallel reads)
- **Delta Lake** basics: write, read, register as SQL table, `DESCRIBE HISTORY`
- **DBFS paths, mounts, and Unity Catalog Volumes** — plus clear **Community Edition alternatives** for each
- **10 hands-on practice questions with full solutions**

### 🔜 Coming in Part 3: Transformations (Core)
select/selectExpr, filter, withColumn deep-dive, string functions, date/time functions, null handling,
and dozens more transformation functions — with the same style: full examples + practice questions.